# Global Field Mechanism — Dissertation Evaluation Notebook
**Conditions:** Passive / Probe / Active

This notebook acts as the primary evaluation orchestrator:
1. Loads JSON results from experiment directories.
2. Structures data into the `RESULTS` format required by `dissertation_plots.py`.
3. Executes hypothesis-driven visualizations (H1, H2, H3) for the dissertation.

## 0. Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import dissertation_plots as dp  # Import your modular plotting script

# Results storage conforming to dissertation_plots structure
RESULTS = {}

## 1. Configuration
Update these paths to match your local directory structure.

In [ ]:
BASE_DIR = '/home/casper/Documents/A_Casper/Brein/CS_Ai_Wolfhampton/Dissertation'

CONFIGS = {
    'CIFAR-Seq-H64': {
        'folder': f'{BASE_DIR}/CIFAR/h64',
        'json': {
            'Active':  'PROBE_RUN_ACTIVE.json',
            'Passive': 'PROBE_RUN_PASSIVE.json',
            'Probe':   'PROBE_RUN_PROBE.json',
        }
    }
    # Add more configurations (e.g., sMNIST-H32) here following the same pattern
}

CONDITIONS = ['Passive', 'Probe', 'Active']

## 2. Data Loading & Extraction
Bridges the JSON logs to the `RESULTS` dictionary format.

In [ ]:
def extract_to_results_schema(data):
    """Converts raw JSON data into the schema required by dissertation_plots.py"""
    # Note: Structure depends on whether you used the Sobol worker or the Probing worker
    # This mapping assumes the 'epochs' list contains the metric snapshots
    hist = data['epochs']
    
    return {
        'epochs':         np.array([e['epoch'] for e in hist]),
        'loss':           np.array([e['loss'] for e in hist]),
        'val_acc':        np.array([e['acc'] for e in hist]) * 100,
        'test_acc':       data.get('test_acc', data.get('acc', 0)) * 100,
        'effective_rank': np.array([e['rank'] for e in hist]),
        'synchrony':      np.array([e['sync'] for e in hist]),
        'interference':   np.array([e['intf'] for e in hist]),
        'a_corr':         np.array([e['acorr'] for e in hist]),
        'entropy':        np.array([e['entr'] for e in hist]),
        'config':         data['parameters']
    }

for cfg_key, cfg_val in CONFIGS.items():
    RESULTS[cfg_key] = {}
    for cond in CONDITIONS:
        fpath = os.path.join(cfg_val['folder'], cfg_val['json'][cond])
        if os.path.exists(fpath):
            with open(fpath, 'r') as f:
                raw_data = json.load(f)
                RESULTS[cfg_key][cond] = extract_to_results_schema(raw_data)
                print(f"Loaded {cfg_key} [{cond}]")
        else:
            print(f"[MISSING] {fpath}")
            RESULTS[cfg_key][cond] = None

## 3. Hypothesis 1: Representational Rigidity

In [ ]:
target_cfg = 'CIFAR-Seq-H64'

dp.plot_rank_stability(RESULTS, target_cfg)
dp.plot_singular_value_spectra(RESULTS, target_cfg, epoch_snapshots=[1, 5, 10, 20])
dp.plot_rank_auc_jitter_comparison(RESULTS)

## 4. Hypothesis 2: Optimization Stability

In [ ]:
dp.plot_convergence_speed(RESULTS, target_cfg)
dp.plot_loss_smoothness(RESULTS, target_cfg)
dp.plot_accuracy_gap_over_time(RESULTS, target_cfg)
dp.plot_temporal_anchor_evidence(RESULTS, target_cfg)

## 5. Hypothesis 3: Coordination Without Collapse

In [ ]:
dp.plot_coordination_trajectory(RESULTS, target_cfg)
dp.plot_metric_dissociation_radar(RESULTS, target_cfg, epoch_idx=-1)  # Final epoch snapshot

## 6. General Dissertation Results Summary

In [ ]:
dp.plot_multi_benchmark_heatmap(RESULTS)
dp.plot_condition_delta_summary(RESULTS)